# Simple Python simulation smoke test

This notebook exercises the public Python API used by the examples: default `VERBOSITY=3` output columns, explicit verbosity selection, and forward source annotations.


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

def find_repo_root(start):
    for path in [start, *start.parents]:
        if (path / "CMakeLists.txt").exists() and (path / "build").exists():
            return path
    return start

repo_root = find_repo_root(Path.cwd().resolve())
build_dir = repo_root / "build"
if build_dir.exists():
    sys.path.insert(0, str(build_dir))

import genulens


## Default result

The default Python output uses the `VERBOSITY=3` style labels and includes relative proper-motion components.


In [2]:
cfg = genulens.Config(l=1.0, b=-3.9, n_simu=2_000, seed=42)

result = genulens.simulate(cfg)
df = pd.DataFrame(result.to_numpy(), columns=result.columns)

required = [
    "wtj",
    "M_L",
    "D_L",
    "D_S",
    "t_E",
    "theta_E",
    "pi_E",
    "pi_EN",
    "pi_EE",
    "mu_rel",
    "mu_rel_N",
    "mu_rel_E",
    "mu_Sl",
    "mu_Sb",
    "I_L",
    "K_L",
    "iS",
    "iL",
    "fREM",
]
missing = [name for name in required if name not in df.columns]
assert not missing, missing
assert len(df) == cfg.n_simu
assert np.all(np.isfinite(df[required].to_numpy()))

df[["wtj", "M_L", "D_L", "D_S", "t_E", "theta_E", "pi_EN", "pi_EE", "mu_rel_N", "mu_rel_E"]].head()


,wtj,M_L,D_L,D_S,t_E,theta_E,pi_EN,pi_EE,mu_rel_N,mu_rel_E
0,0.439495,0.727797,6689.670842,7531.261836,29.332058,0.314656,0.040114,0.034773,2.960630,2.566463
1,0.770385,0.082821,7155.889641,8475.373225,4.581636,0.121137,0.175719,0.037129,9.448504,1.996428
2,0.355760,0.003498,6869.181582,7382.425669,1.353492,0.016980,-0.020661,0.595679,-0.158842,4.579523
3,0.578160,0.255149,4020.146297,7835.624133,38.077042,0.501684,0.240489,0.021367,4.793466,0.425897
4,0.800762,1.006170,7757.517413,8214.310840,18.598094,0.242362,-0.027322,-0.011329,-4.396800,-1.823083


## Explicit verbosity result

The default is already `VERBOSITY=3` style. This cell checks that another verbosity setting still returns its corresponding CLI-style labels.


In [3]:
cfg_v1 = genulens.Config(l=1.0, b=-3.9, n_simu=1_000, seed=43)
cfg_v1.sampling.verbosity = 1

result_v1 = genulens.simulate(cfg_v1)
df_v1 = pd.DataFrame(result_v1.to_numpy(), columns=result_v1.columns)

expected = [
    "wtj",
    "tE",
    "thetaE",
    "piE",
    "M_L",
    "D_S",
    "D_L",
    "mu_rel",
    "iS",
    "iL",
    "tau_s",
    "tau_l",
    "fREM",
]
assert result_v1.columns == expected
assert np.all(np.isfinite(df_v1[["wtj", "M_L", "D_S", "D_L", "mu_rel"]].to_numpy()))

df_v1[expected].head()


,wtj,tE,thetaE,piE,M_L,D_S,D_L,mu_rel,iS,iL,tau_s,tau_l,fREM
0,0.179267,34.908325,0.576570,0.177791,0.398207,6050.831655,3734.469199,6.032717,8.0,6.0,9.574561,8.656024,0.0
1,0.722854,4.475763,0.118197,0.043445,0.334070,8076.535500,7754.914028,9.645610,8.0,8.0,8.116745,8.374130,0.0
2,0.832708,9.538321,0.136683,0.149905,0.111961,8783.055015,7443.520480,5.233997,8.0,8.0,9.774952,10.367646,0.0
3,1.764391,17.647129,0.558801,0.139539,0.491731,8102.778911,4965.509190,11.565731,8.0,4.0,10.103549,4.068387,0.0
4,0.912283,4.061005,0.120921,0.030477,0.487186,8153.127974,7915.298547,10.875703,8.0,8.0,8.114960,7.453657,0.0


## Forward source annotations

Forward source mode appends intrinsic source properties from the shared isochrone lookup. It is an annotation mode and does not replace the legacy LF/CMF event selection.


In [4]:
cfg_src = genulens.Config(l=1.0, b=-3.9, n_simu=1_000, seed=44)
cfg_src.forward_source.enabled = 1
cfg_src.forward_source.photometry = "roman"
cfg_src.forward_source.min_initial_mass_msun = 0.1
cfg_src.forward_source.max_initial_mass_msun = 1.0

result_src = genulens.simulate(cfg_src)
df_src = pd.DataFrame(result_src.to_numpy(), columns=result_src.columns)

source_required = [
    "logage_S",
    "MH_S",
    "M_S_ini",
    "M_S",
    "R_S",
    "teff_S",
    "logg_S",
    "theta_S",
    "M_F146mag_S",
]
missing = [name for name in source_required if name not in df_src.columns]
assert not missing, missing

finite_rows = np.isfinite(df_src[source_required].to_numpy()).all(axis=1)
assert finite_rows.mean() > 0.95

df_src.loc[finite_rows, source_required].head()


,logage_S,MH_S,M_S_ini,M_S,R_S,teff_S,logg_S,theta_S,M_F146mag_S
0,9.937318,0.00,0.119672,0.119672,0.153600,2656.709047,5.140754,0.159609,10.089800
1,9.954243,0.25,0.618435,0.618310,0.620177,3837.139699,4.641730,0.343335,5.688021
2,9.903090,0.00,0.593835,0.593835,0.594054,3917.577581,4.660439,0.378557,5.730977
3,10.000000,0.25,0.283268,0.283210,0.302784,3029.022013,4.925323,0.182629,8.116321
4,10.079181,-0.60,0.165413,0.165607,0.186201,3248.327208,5.113805,0.082469,8.955378
